**MAESTRÍA EN INTELIGENCIA ARTIFICIAL APLICADA**

**Curso:** MLOPs TC5061

Tecnológico de Monterrey

**Alumno** Cristhian Rikardo Hernandez Hernandez

**Docente** Dr. Iván Reyes Amezcua

**Actividad**
Dataset - Notebook | Individual


---

# Exploración de datos: *CDC Diabetes Health Indicators*

Este notebook se usa **únicamente para exploración**: entender el dataset, analizar la calidad de los datos y documentar con evidencia las decisiones de limpieza.

La carga de datos, la limpieza, el entrenamiento y la evaluación del modelo viven en el paquete `diabetes_mlops_v2` y se ejecutan desde la línea de comandos:

```bash
python -m diabetes_mlops_v2.modeling.train
```

Para no duplicar lógica, este notebook **importa** del paquete la carga de datos (`dataset.cargar_datos`), la definición de variables y el esquema (`config`), la paleta de gráficas (`plots`) y la función de limpieza (`dataset.limpiar_datos`).

## 1. Carga y descripción del dataset

In [ ]:
import pandas as pd

from diabetes_mlops_v2.dataset import cargar_datos

# Carga la copia local del dataset (solo la primera vez se descarga desde UCI)
df = cargar_datos()

In [ ]:
print(f"Número de registros: {df.shape[0]:,}")
print(f"Número de variables: {df.shape[1]} ({df.shape[1] - 1} predictoras + 1 objetivo)")
print(f"Valores faltantes totales: {df.isna().sum().sum()}")
print()

# Resumen de estructura: tipo de dato, valores únicos y rango de cada variable
estructura = pd.DataFrame({
    "tipo_pandas": df.dtypes.astype(str),
    "valores_unicos": df.nunique(),
    "min": df.min(),
    "max": df.max(),
})
estructura

### Descripción del dataset: *CDC Diabetes Health Indicators* (UCI id 891)

**Fuente:** Encuesta BRFSS 2014 de los CDC (Centers for Disease Control and Prevention, EE. UU.). Cada fila representa a una persona encuestada.

**Estructura general**

| Característica | Valor |
|---|---|
| Número de registros | **253,680** |
| Número de variables | **22** (21 predictoras + 1 objetivo) |
| Tipo de dato en pandas | Todas `int64` |
| Valores faltantes | Ninguno |
| Tipo de problema | Clasificación binaria |
| Balance de la variable objetivo | 86.1 % sin diabetes (0) · 13.9 % prediabetes/diabetes (1) → **desbalanceado** |

**Variables, tipo y significado**

| # | Variable | Rol | Tipo | Rango | Significado |
|---|---|---|---|---|---|
| 1 | `Diabetes_binary` | Objetivo | Binaria | 0–1 | 0 = sin diabetes, 1 = prediabetes o diabetes |
| 2 | `HighBP` | Predictora | Binaria | 0–1 | Presión arterial alta (1 = sí) |
| 3 | `HighChol` | Predictora | Binaria | 0–1 | Colesterol alto (1 = sí) |
| 4 | `CholCheck` | Predictora | Binaria | 0–1 | Se revisó el colesterol en los últimos 5 años (1 = sí) |
| 5 | `BMI` | Predictora | Entera (numérica continua) | 12–98 | Índice de masa corporal |
| 6 | `Smoker` | Predictora | Binaria | 0–1 | Ha fumado al menos 100 cigarrillos en su vida (1 = sí) |
| 7 | `Stroke` | Predictora | Binaria | 0–1 | Alguna vez le diagnosticaron un derrame cerebral (1 = sí) |
| 8 | `HeartDiseaseorAttack` | Predictora | Binaria | 0–1 | Enfermedad coronaria o infarto al miocardio (1 = sí) |
| 9 | `PhysActivity` | Predictora | Binaria | 0–1 | Actividad física en los últimos 30 días, sin contar el trabajo (1 = sí) |
| 10 | `Fruits` | Predictora | Binaria | 0–1 | Consume fruta 1 o más veces al día (1 = sí) |
| 11 | `Veggies` | Predictora | Binaria | 0–1 | Consume verduras 1 o más veces al día (1 = sí) |
| 12 | `HvyAlcoholConsump` | Predictora | Binaria | 0–1 | Consumo alto de alcohol (hombres >14 bebidas/semana, mujeres >7) (1 = sí) |
| 13 | `AnyHealthcare` | Predictora | Binaria | 0–1 | Cuenta con algún tipo de seguro/cobertura médica (1 = sí) |
| 14 | `NoDocbcCost` | Predictora | Binaria | 0–1 | En los últimos 12 meses no pudo ir al médico por el costo (1 = sí) |
| 15 | `GenHlth` | Predictora | Ordinal | 1–5 | Salud general autopercibida: 1 = excelente … 5 = mala |
| 16 | `MentHlth` | Predictora | Entera (conteo) | 0–30 | Días con mala salud mental en los últimos 30 días |
| 17 | `PhysHlth` | Predictora | Entera (conteo) | 0–30 | Días con mala salud física en los últimos 30 días |
| 18 | `DiffWalk` | Predictora | Binaria | 0–1 | Dificultad seria para caminar o subir escaleras (1 = sí) |
| 19 | `Sex` | Predictora (demográfica) | Binaria | 0–1 | 0 = mujer, 1 = hombre |
| 20 | `Age` | Predictora (demográfica) | Ordinal | 1–13 | Categoría de edad en rangos de 5 años: 1 = 18–24 … 9 = 60–64 … 13 = 80 o más |
| 21 | `Education` | Predictora (demográfica) | Ordinal | 1–6 | Nivel educativo: 1 = nunca asistió a la escuela … 6 = 4+ años de universidad |
| 22 | `Income` | Predictora (demográfica) | Ordinal | 1–8 | Ingreso anual: 1 = menos de \$10,000 … 8 = \$75,000 o más |

**Observaciones**
- Aunque pandas almacena todo como `int64`, la mayoría de las variables son **categóricas codificadas**: 14 binarias, 4 ordinales (`GenHlth`, `Age`, `Education`, `Income`) y solo 3 numéricas propiamente dichas (`BMI`, `MentHlth`, `PhysHlth`).
- La columna `ID` aparece en los metadatos pero no se incluye en `X`, por lo que no hay que eliminarla.
- El desbalance de clases (≈86/14) debe tomarse en cuenta al elegir métricas (p. ej. F1, recall o ROC-AUC en lugar de solo *accuracy*).
- `Sex`, `Income` y `Education` son atributos sensibles según los metadatos del dataset.

## 2. Análisis exploratorio (EDA)

In [ ]:
# Estadísticas descriptivas básicas de todas las variables (transpuesta para leerla por fila)
df.describe().T

### 2.1 Configuración del análisis

Agrupamos las variables según su **tipo real** (no según el `int64` de pandas), porque cada tipo requiere estadísticas y gráficos distintos:

- **Numéricas** (`BMI`, `MentHlth`, `PhysHlth`): media, mediana, dispersión, asimetría, outliers → histogramas y boxplots.
- **Binarias** (14 variables 0/1): la media es directamente la **prevalencia** (% de personas con valor 1).
- **Ordinales** (`GenHlth`, `Age`, `Education`, `Income`): frecuencias por nivel.

Colores fijos en todo el EDA: **azul = sin diabetes (0)** y **naranja = prediabetes/diabetes (1)**.

In [ ]:
# Librerías para cálculo y visualización
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# Definición de variables y paleta compartidas con el código de entrenamiento
from diabetes_mlops_v2.config import BINARIAS, ESQUEMA, NUMERICAS, ORDINALES, TARGET
from diabetes_mlops_v2.plots import (
    AZUL, CMAP_DIVERGENTE, CMAP_SECUENCIAL, ETIQUETA_CLASE, GRIS, NARANJA, PALETA_CLASE, configurar_estilo,
)

# Estilo general de las gráficas
configurar_estilo()

# Tasa global de diabetes, se usa como referencia en varias gráficas
TASA_GLOBAL = df[TARGET].mean()
print(f"Binarias ({len(BINARIAS)}): {BINARIAS}")
print(f"Ordinales ({len(ORDINALES)}): {ORDINALES}")
print(f"Numéricas ({len(NUMERICAS)}): {NUMERICAS}")
print(f"Tasa global de diabetes: {TASA_GLOBAL:.1%}")

### 2.2 Variable objetivo: `Diabetes_binary`

Lo primero es confirmar el balance de clases, porque condiciona la elección de métricas y la estrategia de partición (estratificada).

In [ ]:
# Conteo y porcentaje de registros por clase
conteo = df[TARGET].value_counts().sort_index()
resumen_objetivo = pd.DataFrame({
    "clase": [ETIQUETA_CLASE[i] for i in conteo.index],
    "registros": conteo.values,
    "porcentaje": (conteo / len(df) * 100).round(2).values,
})
display(resumen_objetivo)

# Gráfica de barras horizontales con el tamaño de cada clase
fig, ax = plt.subplots(figsize=(7, 2.6))
ax.barh([ETIQUETA_CLASE[i] for i in conteo.index], conteo.values,
        color=[PALETA_CLASE[i] for i in conteo.index], height=0.55)
# Etiqueta con el conteo y porcentaje al final de cada barra
for i, v in enumerate(conteo.values):
    ax.text(v + 2000, i, f"{v:,}  ({v / len(df):.1%})", va="center", color=GRIS)
ax.set_xlim(0, conteo.max() * 1.3)
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
ax.set_title("Distribución de la variable objetivo")
ax.set_xlabel("Número de registros")
ax.invert_yaxis()
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

### 2.3 Estadísticas descriptivas de las variables numéricas

Además de `describe()`, calculamos **asimetría** (skew), **curtosis**, el **% de ceros** y el número de **outliers según la regla IQR** (valores fuera de Q1 − 1.5·IQR y Q3 + 1.5·IQR).

In [ ]:
# Calcula estadísticas descriptivas extendidas para una variable numérica
def resumen_numerico(serie):
    # Límites de la regla IQR para detectar outliers
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = ((serie < lim_inf) | (serie > lim_sup)).sum()
    return pd.Series({
        "media": serie.mean(), "mediana": serie.median(), "moda": serie.mode()[0],
        "desv_std": serie.std(), "min": serie.min(), "Q1": q1, "Q3": q3, "max": serie.max(),
        "IQR": iqr, "asimetria": serie.skew(), "curtosis": serie.kurt(),
        "pct_ceros": (serie == 0).mean() * 100,
        "lim_sup_IQR": lim_sup, "n_outliers_IQR": outliers,
        "pct_outliers_IQR": outliers / len(serie) * 100,
    })

# Aplica el resumen a cada variable numérica
desc_numericas = df[NUMERICAS].apply(resumen_numerico).T.round(2)
desc_numericas

In [ ]:
# Cuadrícula: histogramas arriba y boxplots abajo, una columna por variable
fig, axes = plt.subplots(2, 3, figsize=(15, 7), gridspec_kw={"height_ratios": [3, 1]}, sharex="col")
bins = {"BMI": np.arange(12, 100, 1), "MentHlth": np.arange(0, 32, 1), "PhysHlth": np.arange(0, 32, 1)}
nombres = {"BMI": "Índice de masa corporal", "MentHlth": "Días con mala salud mental (últ. 30)",
           "PhysHlth": "Días con mala salud física (últ. 30)"}

for j, col in enumerate(NUMERICAS):
    # Histograma con líneas de media y mediana
    ax = axes[0, j]
    ax.hist(df[col], bins=bins[col], color=AZUL, edgecolor="white", linewidth=0.5)
    ax.axvline(df[col].mean(), color=NARANJA, lw=2, label=f"media = {df[col].mean():.1f}")
    ax.axvline(df[col].median(), color=GRIS, lw=2, ls="--", label=f"mediana = {df[col].median():.0f}")
    ax.set_title(f"{col}\n{nombres[col]}")
    ax.set_ylabel("Frecuencia")
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
    ax.legend(frameon=False)

    # Boxplot para visualizar dispersión y outliers
    sns.boxplot(x=df[col], ax=axes[1, j], color="#b7d3f6", width=0.5, linewidth=1,
                flierprops=dict(marker="o", markersize=2, alpha=0.2, markerfacecolor=GRIS, markeredgecolor="none"))
    axes[1, j].set_xlabel(col)

plt.suptitle("Distribución de las variables numéricas", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.show()

### 2.4 Variables binarias: prevalencia

Para una variable 0/1 la media es la proporción de personas que responden "sí". Comparamos la prevalencia en la población total.

In [ ]:
# Prevalencia = % de registros con valor 1 en cada variable binaria
prevalencia = (df[BINARIAS].mean() * 100).sort_values()

# Gráfica de barras horizontales ordenada de menor a mayor
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(prevalencia.index, prevalencia.values, color=AZUL, height=0.6)
for i, v in enumerate(prevalencia.values):
    ax.text(v + 1, i, f"{v:.1f}%", va="center", color=GRIS, fontsize=9)
ax.set_xlim(0, 110)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title("Prevalencia de cada variable binaria (% con valor = 1)")
ax.set_xlabel("% de la población")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

### 2.5 Variables ordinales: distribución por nivel

In [ ]:
# Etiquetas legibles para cada nivel de las variables ordinales
etiquetas_ordinales = {
    "GenHlth": {1: "Excelente", 2: "Muy buena", 3: "Buena", 4: "Regular", 5: "Mala"},
    "Age": {1: "18-24", 2: "25-29", 3: "30-34", 4: "35-39", 5: "40-44", 6: "45-49", 7: "50-54",
            8: "55-59", 9: "60-64", 10: "65-69", 11: "70-74", 12: "75-79", 13: "80+"},
    "Education": {1: "Ninguna", 2: "Primaria", 3: "Sec. inc.", 4: "Preparatoria", 5: "Univ. inc.", 6: "Universidad"},
    "Income": {1: "<10k", 2: "10-15k", 3: "15-20k", 4: "20-25k", 5: "25-35k", 6: "35-50k", 7: "50-75k", 8: "75k+"},
}

# Una gráfica de barras por variable con el % de registros en cada nivel
fig, axes = plt.subplots(2, 2, figsize=(15, 8))
for ax, col in zip(axes.flat, ORDINALES):
    pct = df[col].value_counts(normalize=True).sort_index() * 100
    etiquetas = [etiquetas_ordinales[col][k] for k in pct.index]
    ax.bar(etiquetas, pct.values, color=AZUL, width=0.7)
    for x, v in enumerate(pct.values):
        ax.text(x, v + 0.5, f"{v:.1f}%", ha="center", color=GRIS, fontsize=8)
    ax.set_title(col)
    ax.set_ylabel("% de registros")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
    ax.tick_params(axis="x", rotation=45 if col in ("Age", "Education") else 0, labelsize=9)
    ax.grid(axis="x", visible=False)
plt.suptitle("Distribución de las variables ordinales", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.show()

## Relaciones entre variables

### 2.6 Variables numéricas vs. variable objetivo

In [ ]:
# Media, mediana y desviación estándar de cada variable numérica por clase
df.groupby(TARGET)[NUMERICAS].agg(["mean", "median", "std"]).T.rename(columns=ETIQUETA_CLASE).round(2)

In [ ]:
# Boxplots de cada variable numérica separados por clase (el rombo blanco marca la media)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, col in zip(axes, NUMERICAS):
    sns.boxplot(data=df, x=TARGET, y=col, hue=TARGET, palette=PALETA_CLASE, ax=ax, legend=False,
                width=0.5, linewidth=1, showmeans=True,
                meanprops=dict(marker="D", markerfacecolor="white", markeredgecolor="black", markersize=6),
                flierprops=dict(marker="o", markersize=2, alpha=0.15, markeredgecolor="none"))
    ax.set_xticks([0, 1], [ETIQUETA_CLASE[0], ETIQUETA_CLASE[1]])
    ax.set_xlabel("")
    ax.set_title(col)
plt.suptitle("Variables numéricas por clase (rombo blanco = media)", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.show()

# Densidad del BMI por clase: muestra el desplazamiento de toda la distribución, no solo de la mediana
fig, ax = plt.subplots(figsize=(10, 4))
for clase in (0, 1):
    sns.kdeplot(df.loc[df[TARGET] == clase, "BMI"], ax=ax, color=PALETA_CLASE[clase],
                fill=True, alpha=0.25, linewidth=2, label=ETIQUETA_CLASE[clase], clip=(12, 70))
ax.set_xlim(12, 70)
ax.set_ylabel("Densidad")
ax.set_title("Densidad del BMI por clase (normalizada dentro de cada clase)")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

### 2.7 Variables binarias vs. variable objetivo

Para cada factor binario comparamos la **tasa de diabetes** entre quienes tienen el factor (=1) y quienes no (=0). El cociente entre ambas es el **riesgo relativo (RR)**: RR > 1 indica asociación positiva con diabetes, RR < 1 asociación negativa.

In [ ]:
# Tasa de diabetes cuando cada variable binaria vale 0 y cuando vale 1, y su riesgo relativo
filas = []
for col in BINARIAS:
    tasas = df.groupby(col)[TARGET].mean()
    filas.append({"variable": col, "tasa_si_0": tasas[0] * 100, "tasa_si_1": tasas[1] * 100,
                  "riesgo_relativo": tasas[1] / tasas[0]})
tasas_binarias = pd.DataFrame(filas).set_index("variable").sort_values("riesgo_relativo")
display(tasas_binarias.round(2).sort_values("riesgo_relativo", ascending=False))

# Gráfico de "mancuernas": cada fila une la tasa con factor=0 y factor=1
fig, ax = plt.subplots(figsize=(10, 7))
y = np.arange(len(tasas_binarias))
ax.hlines(y, tasas_binarias["tasa_si_0"], tasas_binarias["tasa_si_1"], color="#c3c2b7", lw=2)
ax.scatter(tasas_binarias["tasa_si_0"], y, color=GRIS, s=60, zorder=3, label="Variable = 0 (no)")
ax.scatter(tasas_binarias["tasa_si_1"], y, color=NARANJA, s=60, zorder=3, label="Variable = 1 (sí)")
# Etiqueta con el riesgo relativo a la derecha de cada fila
for yi, (_, r) in zip(y, tasas_binarias.iterrows()):
    ax.text(max(r["tasa_si_0"], r["tasa_si_1"]) + 0.8, yi, f"RR {r['riesgo_relativo']:.2f}",
            va="center", fontsize=9, color=GRIS)
# Línea de referencia con la tasa global
ax.axvline(TASA_GLOBAL * 100, color=AZUL, ls="--", lw=1.2, label=f"Tasa global ({TASA_GLOBAL:.1%})")
ax.set_yticks(y, tasas_binarias.index)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_xlim(0, tasas_binarias[["tasa_si_0", "tasa_si_1"]].max().max() + 7)
ax.set_xlabel("% con prediabetes/diabetes")
ax.set_title("Tasa de diabetes según cada variable binaria")
ax.legend(frameon=False, loc="lower right")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

### 2.8 Variables ordinales vs. variable objetivo

Tasa de diabetes en cada nivel. La línea punteada es la tasa global (≈13.9 %).

In [ ]:
# Tasa de diabetes en cada nivel de las variables ordinales
fig, axes = plt.subplots(2, 2, figsize=(15, 8))
for ax, col in zip(axes.flat, ORDINALES):
    tasa = df.groupby(col)[TARGET].mean() * 100
    etiquetas = [etiquetas_ordinales[col][k] for k in tasa.index]
    ax.plot(etiquetas, tasa.values, color=NARANJA, lw=2, marker="o", markersize=8,
            markeredgecolor="white", markeredgewidth=2)
    # Línea de referencia con la tasa global
    ax.axhline(TASA_GLOBAL * 100, color=AZUL, ls="--", lw=1.2)
    # Etiquetamos solo extremos para no saturar el gráfico
    for x in (0, len(tasa) - 1):
        ax.annotate(f"{tasa.values[x]:.1f}%", (x, tasa.values[x]), textcoords="offset points",
                    xytext=(0, 10), ha="center", fontsize=9, color=GRIS)
    ax.set_title(col)
    ax.set_ylabel("% con diabetes")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
    ax.set_ylim(0, tasa.max() * 1.25)
    ax.tick_params(axis="x", rotation=45 if col in ("Age", "Education") else 0, labelsize=9)
plt.suptitle("Tasa de diabetes por nivel de cada variable ordinal", fontweight="bold", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# BMI discretizado según categorías de la OMS: permite ver la relación como tasa y no solo como distribución
cortes = [0, 18.5, 25, 30, 35, 40, 100]
nombres_bmi = ["Bajo peso\n<18.5", "Normal\n18.5-25", "Sobrepeso\n25-30", "Obesidad I\n30-35",
               "Obesidad II\n35-40", "Obesidad III\n40+"]
bmi_cat = pd.cut(df["BMI"], bins=cortes, labels=nombres_bmi, right=False)

# Número de registros y tasa de diabetes por categoría
tabla_bmi = df.groupby(bmi_cat, observed=True)[TARGET].agg(registros="size", tasa="mean")
tabla_bmi["tasa"] *= 100
display(tabla_bmi.rename(index=lambda s: s.replace("\n", " ")).round(2))

# Gráfica de barras con la tasa por categoría y la tasa global como referencia
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.bar(tabla_bmi.index.astype(str), tabla_bmi["tasa"], color=NARANJA, width=0.65)
for x, v in enumerate(tabla_bmi["tasa"]):
    ax.text(x, v + 0.8, f"{v:.1f}%", ha="center", color=GRIS)
ax.axhline(TASA_GLOBAL * 100, color=AZUL, ls="--", lw=1.2, label=f"Tasa global ({TASA_GLOBAL:.1%})")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_ylabel("% con diabetes")
ax.set_title("Tasa de diabetes por categoría de BMI (OMS)")
ax.legend(frameon=False)
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

### 2.9 Matriz de correlación

Usamos la **correlación de Pearson (r)** para medir la fuerza y dirección de la relación lineal entre cada par de variables, incluida la variable objetivo. Aunque la mayoría de las variables son binarias u ordinales, Pearson sigue siendo interpretable en este caso: entre dos variables binarias equivale al **coeficiente phi** y entre una binaria y una numérica equivale a la **correlación punto-biserial**. Valores cercanos a 1 indican relación fuerte y cercanos a 0, relación lineal débil o nula.

In [ ]:
# Matriz de correlación de Pearson entre todas las variables
corr = df.corr(method="pearson")

# Mapa de calor: rojo = correlación positiva, azul = negativa
fig, ax = plt.subplots(figsize=(15, 12))
sns.heatmap(
    corr,
    cmap=CMAP_DIVERGENTE,
    vmin=-1,
    vmax=1,
    center=0,
    annot=True,
    fmt=".2f",  # Coeficiente con dos decimales (r va de -1 a 1, no es un porcentaje)
    annot_kws={"size": 7},
    linewidths=1,
    linecolor="white",
    cbar_kws={"label": "Correlación de Pearson (r)", "shrink": 0.7},
    ax=ax,
)

ax.set_title("Matriz de correlación de Pearson", fontsize=14)
ax.grid(False)
plt.tight_layout()
plt.show()

### 2.10 Hallazgos principales del EDA

**Distribución de las variables**
- **Objetivo desbalanceado:** 86.1 % sin diabetes vs. 13.9 % con prediabetes/diabetes. En el modelado habrá que usar partición **estratificada** y métricas como recall, F1 o ROC-AUC; *accuracy* sola sería engañosa (un modelo que siempre predice 0 tendría 86 % de accuracy).
- **BMI:** media 28.4 y mediana 27 → la persona típica está en **sobrepeso**. Tiene sesgo a la derecha (asimetría 2.1, curtosis 11) y ~3.9 % de outliers por IQR (> 41.5), con máximos de 98 que son biológicamente improbables.
- **MentHlth y PhysHlth:** distribuciones muy concentradas en cero (69 % y 63 % de ceros) con un segundo pico en 30 días. La regla IQR marca 14–16 % de "outliers", pero **no son errores**: son respuestas válidas de personas con mala salud crónica, por lo que no conviene eliminarlas.
- **Binarias:** algunas están casi constantes — `CholCheck` (96 % = 1), `AnyHealthcare` (95 %), `HvyAlcoholConsump` (6 %) y `Stroke` (4 %) — y aportan poca variabilidad.
- **Ordinales:** la muestra está cargada hacia edades de 55–69 años, ingresos altos (la categoría modal es ≥ \$75k) y alta escolaridad.

**Relaciones con la variable objetivo**
- **Salud general (`GenHlth`)** es la predictora individual más correlacionada con el objetivo (r = 0.29): la tasa de diabetes pasa de **2.5 %** con salud "excelente" a **37.9 %** con salud "mala".
- **Presión alta (`HighBP`)**: 24.5 % de diabetes vs. 6.0 % sin ella (RR ≈ 4). `HighChol`, `DiffWalk`, `HeartDiseaseorAttack` y `Stroke` casi triplican el riesgo.
- **BMI:** relación monótona y fuerte: 5.7 % con peso normal → 33.5 % en obesidad III. La distribución completa del BMI en diabéticos está desplazada ~4 puntos a la derecha.
- **Edad:** la tasa crece de 1.4 % (18–24) a ~22 % (70–74) y baja ligeramente en 80+ (posible sesgo de supervivencia).
- **Factores socioeconómicos:** a mayor ingreso y escolaridad, menor tasa (ingreso < \$15k ≈ 25 % vs. ≥ \$75k = 8 %).
- **Factores protectores:** `PhysActivity` (RR 0.55), `Veggies` y `Fruits` se asocian con menor tasa. `HvyAlcoholConsump` también aparece como "protector" (RR 0.40), lo cual es contraintuitivo y probablemente refleja **confusión** (p. ej. edad o autoselección), no causalidad.
- `Sex`, `AnyHealthcare` y `NoDocbcCost` tienen una correlación prácticamente nula con el objetivo (|r| < 0.04), y `MentHlth` muy débil (r = 0.07).

**Relaciones entre predictoras**
- Las correlaciones más altas entre predictoras son moderadas (|r| ≤ 0.52): `GenHlth`–`PhysHlth` (0.52), `DiffWalk`–`PhysHlth` (0.48), `GenHlth`–`DiffWalk` (0.46) e `Income`–`Education` (0.45). Forman un bloque de "estado de salud / nivel socioeconómico", pero **no hay multicolinealidad severa**, así que un modelo lineal base (regresión logística) es viable sin eliminar variables.

## 3. Calidad de los datos y decisiones de limpieza

### Criterios generales de limpieza

1. **Cada decisión se basa en evidencia** de los datos o en conocimiento del dominio, no solo en una regla automática. Esta sección del notebook documenta esa evidencia.
2. **La limpieza se implementa una sola vez**, en la función `diabetes_mlops_v2.dataset.limpiar_datos`, con sus parámetros en `diabetes_mlops_v2.config.ConfigLimpieza`. El script de entrenamiento y este notebook usan la misma función.
3. **Sin fuga de información (*data leakage*):** los umbrales de limpieza son **fijos** (definidos por dominio), no se calculan con estadísticas del dataset completo. Las transformaciones que sí aprenden de los datos (por ejemplo, el escalado) se ajustan **solo con el conjunto de entrenamiento** dentro del `Pipeline` del modelo.

### 3.1 Valores faltantes y valores fuera de dominio

En encuestas como BRFSS, los datos faltantes no siempre aparecen como `NaN`: a veces se codifican como `7`/`9` ("no sabe"/"se negó a responder") o `77`/`99`. Por eso, además de contar los `NaN`, validamos que **cada variable esté dentro de su dominio documentado** (esquema esperado).

In [ ]:
# Conteo de valores nulos por columna
nulos = df.isna().sum()
print(f"Valores NaN en todo el dataset: {nulos.sum()}")

# Revisa cuántos valores caen fuera del rango esperado (ESQUEMA viene de diabetes_mlops_v2.config)
validacion = pd.DataFrame({
    "rango_esperado": {c: f"{lo}-{hi}" for c, (lo, hi) in ESQUEMA.items()},
    "min_observado": df.min(),
    "max_observado": df.max(),
    "fuera_de_dominio": {c: (~df[c].between(lo, hi)).sum() for c, (lo, hi) in ESQUEMA.items()},
    "NaN": nulos,
})
validacion

**Decisión 1. Valores faltantes: no se requiere imputación.**
- No hay `NaN` y **ninguna variable tiene valores fuera de su dominio** (para `BMI`, que no tiene un rango oficial, se valida un rango amplio de captura de 10 a 100; su plausibilidad clínica se analiza en 3.3), así que tampoco hay faltantes "ocultos" con códigos como 7, 9, 77 o 99. El dataset de UCI ya viene preprocesado: los registros con respuestas inválidas se eliminaron antes de publicarlo.
- Aun así, el esquema `ESQUEMA` se conserva como **validación de datos**: en un flujo de MLOps, cualquier dato nuevo (reentrenamiento o inferencia) debe pasar por esta verificación antes de llegar al modelo.

### 3.2 Duplicados

In [ ]:
# Filas idénticas en todas las columnas (incluida la variable objetivo)
mask_dup = df.duplicated()
print(f"Filas duplicadas: {mask_dup.sum():,} ({mask_dup.mean():.2%} del total)")
print(f"Registros únicos: {(~mask_dup).sum():,}")

# Perfiles con las mismas predictoras pero distinta etiqueta
predictoras = df.columns.drop(TARGET)
etiquetas_por_perfil = df.groupby(list(predictoras))[TARGET].nunique()
print(f"Perfiles idénticos con etiquetas contradictorias: {(etiquetas_por_perfil > 1).sum():,}")

# ¿Los duplicados son aleatorios o tienen un perfil particular?
comparacion_dup = pd.DataFrame({
    "duplicados": df[mask_dup][[TARGET, "GenHlth", "Age", "BMI", "HighBP"]].mean(),
    "no_duplicados": df[~mask_dup][[TARGET, "GenHlth", "Age", "BMI", "HighBP"]].mean(),
}).round(3)
display(comparacion_dup)

# Perfiles más repetidos
df[mask_dup].value_counts().head(5).rename("repeticiones").reset_index()

**Decisión 2. Duplicados: se conservan.**

Hay ~24,200 filas idénticas (9.5 %). Antes de eliminarlas hay que preguntarse **si son errores o personas distintas que respondieron igual**. La evidencia apunta a lo segundo:
- El dataset **no tiene identificador de persona** y todas las variables son discretas (la mayoría binarias), así que el número de combinaciones posibles es limitado. Es esperable que miles de personas compartan exactamente el mismo perfil.
- Los duplicados **no son aleatorios**: se concentran en el perfil más común de la población (personas jóvenes con salud excelente o muy buena, sin presión ni colesterol altos, con BMI normal). Su tasa de diabetes es de solo ~1 %, frente a ~15 % en el resto.
- **Eliminarlos sesgaría la muestra:** la prevalencia de diabetes subiría artificialmente de 13.9 % a 15.3 % y se subrepresentaría a la población sana, que es justo la que el modelo encontrará con más frecuencia en producción.

*Riesgo considerado:* algunas filas idénticas quedarán en entrenamiento y en prueba a la vez. No es una fuga de información real, porque en producción esos perfiles también se repiten. La estratificación no evita que filas idénticas caigan en ambos conjuntos; este efecto se acepta porque refleja la frecuencia real de esos perfiles en la población.

Los ~1,500 perfiles con **etiquetas contradictorias** también se conservan: representan ruido irreducible (personas con el mismo perfil de riesgo, unas con diabetes y otras sin ella) y marcan un límite natural al desempeño que puede alcanzar cualquier modelo.

### 3.3 Outliers

En el EDA (sección 2.3) la regla IQR marcó outliers en las tres variables numéricas. La regla IQR solo dice qué valores son **estadísticamente raros**, no cuáles son **erróneos**. Analizamos cada variable por separado.

In [ ]:
# Tasa de diabetes en la cola alta del BMI: si los valores extremos son reales,
# la tasa debería seguir subiendo con el BMI (como ocurre de 25 a 40)
bins_bmi = [12, 40, 45, 50, 55, 60, 65, 70, 75, 80, 90, 98]
tramos_bmi = pd.cut(df["BMI"], bins=bins_bmi, include_lowest=True)
cola_bmi = df.groupby(tramos_bmi, observed=True)[TARGET].agg(registros="size", tasa_diabetes="mean")
cola_bmi["tasa_diabetes"] = (cola_bmi["tasa_diabetes"] * 100).round(1)
display(cola_bmi)

# Gráfica de la tasa de diabetes por tramo de BMI, marcando el umbral de corte
fig, ax = plt.subplots(figsize=(11, 4.5))
etiquetas_tramos = [f"{int(i.left)}-{int(i.right)}" if i.left > 12 else "≤40" for i in cola_bmi.index]
colores = [NARANJA if i.left < 70 else GRIS for i in cola_bmi.index]
ax.bar(etiquetas_tramos, cola_bmi["tasa_diabetes"], color=colores, width=0.65)
for x, (v, n) in enumerate(zip(cola_bmi["tasa_diabetes"], cola_bmi["registros"])):
    ax.text(x, v + 0.8, f"{v:.0f}%\nn={n:,}", ha="center", color=GRIS, fontsize=8)
ax.axvline(6.5, color=GRIS, ls="--", lw=1.2)
ax.text(6.6, cola_bmi["tasa_diabetes"].max() + 2, "BMI > 70: se eliminan", color=GRIS, fontsize=9)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_ylim(0, cola_bmi["tasa_diabetes"].max() + 10)
ax.set_xlabel("Tramo de BMI")
ax.set_ylabel("% con diabetes")
ax.set_title("Tasa de diabetes en la cola alta del BMI")
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

# Valores de BMI por encima de 70: llama la atención que se acumulen en ciertos números
print("Frecuencia de cada valor de BMI > 70:")
print(df.loc[df["BMI"] > 70, "BMI"].value_counts().sort_index().to_dict())

**Decisión 3. BMI: se eliminan los registros con BMI > 70 (≈ 0.2 % de los datos).**
- Entre 40 y 70, la tasa de diabetes **sigue subiendo** (33 % → 44 %), igual que en el resto del rango. Son personas con obesidad severa real y, además, son el grupo **más informativo** para el modelo. **No se tocan**, aunque la regla IQR las marque como outliers (cualquier BMI > 41.5).
- Por encima de 70 la relación **se rompe**: la tasa cae a 10–17 %, el nivel de la población general. Además, los valores se acumulan en números específicos (73, 75, 77, 79, 81, 84, 87…), un patrón típico de **errores de captura o de conversión de unidades** en el peso o la estatura autorreportados con los que BRFSS calcula el BMI. Un BMI > 70 es clínicamente excepcional y no es creíble que aparezca en cientos de encuestados.
- **¿Por qué eliminar y no recortar (*capping*) a 70?** Si el valor es un error, el BMI real es desconocido. Recortarlo a 70 le asignaría a esas personas obesidad extrema, lo cual contradice su tasa de diabetes observada e introduciría ruido. Como son pocos registros (≈ 580), eliminarlos es la opción más limpia.
- El umbral de 70 es **fijo y de dominio**, no un percentil calculado sobre los datos, así que no genera fuga de información y se puede aplicar igual en producción.
- Los valores bajos (BMI 12–14, 68 registros) son extremos pero fisiológicamente posibles (desnutrición severa) y se conservan.

**Decisión 4. `MentHlth` y `PhysHlth`: se conservan sin cambios.**
- La regla IQR marca como outliers 14–16 % de los registros, pero eso se debe a que la mayoría responde 0 días (el IQR es muy estrecho), no a que los valores sean erróneos.
- Todos los valores están dentro del dominio válido (0–30 días). El pico en 30 corresponde a personas con mala salud **crónica**, información clínicamente relevante.
- No se transforman: un baseline de regresión logística con escalado puede usarlas tal cual. Si en el futuro hiciera falta, se podrían discretizar (0, 1–13, 14–30 días).

### 3.4 Tipos de datos

Todas las columnas vienen como `int64`. No hay tipos "incorrectos" en el sentido de números guardados como texto, pero sí **tipos ineficientes** y **semánticamente imprecisos**.

In [ ]:
# Tipos actuales y memoria utilizada
print("Tipos actuales:", df.dtypes.value_counts().to_dict())
print(f"Memoria con int64: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

# Todas las variables caben en int8 (rango -128 a 127): la mayor es BMI, con máximo 98
print(f"Valor máximo en todo el dataset: {df.max().max()}")
print(f"Memoria con int8: {df.astype('int8').memory_usage(deep=True).sum() / 1e6:.1f} MB")

**Decisión 5. Tipos: convertir todas las columnas a `int8` y mantener la codificación numérica.**
- Todos los valores son enteros entre 0 y 98, así que `int8` los representa sin pérdida y **reduce la memoria 8 veces** (de ~45 MB a ~6 MB). Esto acelera el entrenamiento y abarata el almacenamiento de versiones del dataset.
- **Binarias:** se mantienen como 0/1 enteros, que es lo que esperan los modelos de scikit-learn.
- **Ordinales** (`GenHlth`, `Age`, `Education`, `Income`): se mantienen como códigos numéricos (*ordinal encoding*) porque el orden tiene significado (por ejemplo, la tasa de diabetes crece casi monótonamente con `Age` y `GenHlth`, como se vio en el EDA). Convertirlas a `category` obligaría a aplicar *one-hot encoding* y a perder ese orden, lo cual no es necesario para un modelo base.
- **`BMI`** se queda como entero porque el dataset ya viene redondeado; convertirlo a `float` no agrega información.

### 3.5 Efecto de la limpieza

Las decisiones anteriores están implementadas en `diabetes_mlops_v2.dataset.limpiar_datos`. Aquí solo se aplica esa función para verificar su efecto sobre los datos; no se redefine ninguna lógica.

In [ ]:
from diabetes_mlops_v2.config import ConfigLimpieza
from diabetes_mlops_v2.dataset import limpiar_datos

# Aplica la misma limpieza que usa el script de entrenamiento
config_limpieza = ConfigLimpieza()
df_limpio = limpiar_datos(df, config_limpieza)
config_limpieza

In [ ]:
# Verificaciones del resultado de la limpieza
assert df_limpio["BMI"].max() <= config_limpieza.bmi_max_plausible
assert (df_limpio.dtypes == config_limpieza.tipo_datos).all()

# Comparación antes vs. después
resumen_limpieza = pd.DataFrame({
    "original": {
        "registros": len(df),
        "BMI máximo": df["BMI"].max(),
        "tasa de diabetes (%)": round(df[TARGET].mean() * 100, 2),
        "memoria (MB)": round(df.memory_usage(deep=True).sum() / 1e6, 1),
    },
    "limpio": {
        "registros": len(df_limpio),
        "BMI máximo": df_limpio["BMI"].max(),
        "tasa de diabetes (%)": round(df_limpio[TARGET].mean() * 100, 2),
        "memoria (MB)": round(df_limpio.memory_usage(deep=True).sum() / 1e6, 1),
    },
}, dtype=object)
print(f"Registros eliminados: {len(df) - len(df_limpio):,} ({(len(df) - len(df_limpio)) / len(df):.2%})")
resumen_limpieza

### 3.6 Registro de decisiones

| # | Problema | Evidencia | Decisión | Impacto |
|---|---|---|---|---|
| 1 | Valores faltantes | 0 `NaN` y 0 valores fuera del dominio documentado | No se imputa; el esquema se conserva como validación | Ninguno |
| 2 | Duplicados (~24,200 filas, 9.5 %) | Sin ID de persona; variables discretas; se concentran en el perfil sano más común (~1 % con diabetes) | **Se conservan** | Evita sesgar la prevalencia de 13.9 % a 15.3 % |
| 3 | BMI > 70 (≈ 580 filas) | La tasa de diabetes deja de crecer y cae al nivel de la población general; valores acumulados en números específicos | **Se eliminan** (umbral fijo de dominio) | −0.2 % de registros |
| 3b | BMI 41.5–70 (outliers IQR) | La tasa de diabetes sigue creciendo de forma coherente | Se conservan | Mantiene al grupo de mayor riesgo |
| 4 | `MentHlth` / `PhysHlth` (outliers IQR) | Valores dentro del dominio 0–30; distribución concentrada en 0 | Se conservan sin transformar | Ninguno |
| 5 | Tipos `int64` | Todos los valores son enteros entre 0 y 98 | Conversión a `int8`; ordinales con codificación numérica | Memoria ÷ 8 |

**Siguiente paso:** el entrenamiento, la selección de hiperparámetros, la elección del umbral y la evaluación del modelo se ejecutan con `python -m diabetes_mlops_v2.modeling.train`, que aplica esta misma limpieza.